# Iraq Farms Health — compare our 2026 crop map with ESA's official WorldCereal 2021 (read-only)
For every area we have processed, this fetches ESA's published **WorldCereal 2021** winter-cereal and cropland maps (the same Terrascope service Iraq Farms Health already uses) and compares them, pixel by pixel on a ~16 m grid, with our **2025-26** map. Results by governorate and region, plus a difference layer for the private preview map.

**Read-only and safe to run in a second tab** while batches run. Run it **after** the publish notebook (it reads `publish/2026/` in Drive and adds `compare_2021.json` to it). Takes roughly 10–20 minutes for the current ~27 blocks.

**Like-for-like check:** 2021 is a yes/no "winter cereal" map. Our 2025-26 run picks the most likely of 24 crops. So this notebook also re-counts 2025-26 cereal the 2021 way (wheat + barley + rye + oats probability ≥ 50%), and at ≥ 70%, from the original tiles, and prints how much the result depends on the rule.

In [ ]:
RUN_NAME    = "iraq_winter2026"
DATA_FOLDER = "2026"

from google.colab import drive
drive.mount("/content/drive")
import os, shutil, datetime
RUN = f"/content/drive/MyDrive/iraq-farms-health/worldcereal/{RUN_NAME}"
PUB = f"{RUN}/publish/{DATA_FOLDER}"
OUT = "/content/compare_2021"
print("Published layer found:", os.path.exists(f"{PUB}/index.json"))
!pip -q install rasterio

In [ ]:
%%writefile /content/compare_2021.py
"""Compare our 2025-26 WorldCereal run with ESA's official WorldCereal 2021 product.

python compare_2021.py <run_folder> <publish_folder> <out_folder> [--zoom 13] [--test]

Reads (never changes):
  <run_folder>/plan/grid_all.geojson + batch_*/job_tracking.csv  → which tiles are processed
  <publish_folder>/index.json, cells/*.tif, boundaries.geojson   → our published 2026 layer
  ESA WorldCereal 2021 v100 (classification) via the Terrascope WMTS that Iraq Farms Health already uses:
    wintercereals (wheat/barley/rye) and temporarycrops (annual cropland)
Compares both on the same web-map grid (zoom 13 ≈ 16 m on the ground), only inside processed tiles.
Writes compare_2021.json, compare_2021_governorates.csv and diff/*.png (+ diff/index.json) for the preview map.
"""
import argparse
import concurrent.futures as cf
import csv
import glob
import io
import json
import math
import os
import time
import urllib.request
from collections import defaultdict

import numpy as np
from scipy import ndimage
import rasterio
from PIL import Image
from rasterio.features import rasterize
from rasterio.transform import Affine
from rasterio.warp import Resampling, reproject

R = 20037508.342789244
ap = argparse.ArgumentParser()
ap.add_argument("run"); ap.add_argument("pub"); ap.add_argument("out")
ap.add_argument("--zoom", type=int, default=13)
ap.add_argument("--threads", type=int, default=16)
ap.add_argument("--test", action="store_true", help="fetch one tile per product and stop")
A = ap.parse_args()
Z = A.zoom
os.makedirs(os.path.join(A.out, "diff"), exist_ok=True)

import urllib.parse
WMTS = ("https://wmts.terrascope.be/wmts?service=WMTS&request=GetTile&version=1.0.0"
        "&layer=esa-worldcereal-{p}-10m-2021-v1_classification&style=default&format=image%2Fpng"
        "&tilematrixset=EPSG:3857&TileMatrix={z}&TileRow={y}&TileCol={x}&TIME=2021-01-01"
        "&name=classification&assets=classification&nodata=255&colormap=" + urllib.parse.quote(json.dumps({"100": "#ff0000"})))
PRODUCTS = ["wintercereals", "temporarycrops"]
failures = defaultdict(int)


def fetch(p, z, x, y):
    """-> 256x256 bool mask (True = WorldCereal 2021 says yes), or None after repeated failure."""
    if os.environ.get("WC_FAKE_2021"):          # offline testing only
        rng = np.random.default_rng(x * 7 + y * 13 + len(p)); return rng.random((256, 256)) < (0.3 if p == "wintercereals" else 0.5)
    url = WMTS.format(p=p, z=z, x=x, y=y)
    for attempt in range(4):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "IraqFarmsHealth-validation"}), timeout=60) as r:
                data = r.read()
            if not data:
                return np.zeros((256, 256), bool)
            im = Image.open(io.BytesIO(data)).convert("RGBA")
            a = np.asarray(im)
            return (a[..., 3] > 0) & (a[..., 0] > 128)
        except urllib.error.HTTPError as e:
            if e.code in (204, 404):
                return np.zeros((256, 256), bool)
            time.sleep(1 + attempt * 2)
        except Exception:
            time.sleep(1 + attempt * 2)
    failures[p] += 1
    return None


if A.test:
    # guide field area (Tarmiyah) and an Erbil plain tile
    for lon, lat in ((44.0607, 33.9415), (44.0, 36.2)):
        n = 2 ** Z
        x = int((lon + 180) / 360 * n); y = int((1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n)
        for p in PRODUCTS:
            m = fetch(p, Z, x, y)
            print(f"{p:15s} z{Z}/{x}/{y} near {lat},{lon}: " + ("FAILED" if m is None else f"{m.mean() * 100:.1f}% of pixels = yes"))
    raise SystemExit

# ------------------------------------------------------------------ our published layer
idx = json.load(open(os.path.join(A.pub, "index.json")))
classes = idx["codes"]["classes"]
LESS = idx["codes"]["less_sure_offset"]
WINTER = {i + 1 for i, c in enumerate(classes) if c in ("wheat", "barley", "rye", "oats")}
NOCROP = {i + 1 for i, c in enumerate(classes) if c == "no_crop"}
WHEAT = {i + 1 for i, c in enumerate(classes) if c == "wheat"}
BARLEY = {i + 1 for i, c in enumerate(classes) if c == "barley"}

# processed footprint = boxes of finished tiles
plan = json.load(open(os.path.join(A.run, "plan", "grid_all.geojson")))
done = set()
for b in glob.glob(os.path.join(A.run, "batch_*", "job_tracking.csv")):
    for r in csv.DictReader(open(b, newline="")):
        if r.get("status") == "finished":
            done.add(r["tile_name"])
# only tiles that are in the published map, so the comparison always matches what the page shows
# (the batch keeps running, so tiles can finish between the publish run and this one)
published = {p_["tile"] for p_ in idx.get("processed", [])}
if published:
    late = sorted(done - published)
    if late:
        print(f"Skipping {len(late)} tile(s) finished after the last publish run (re-run publish to include them): {', '.join(late)}")
    done &= published
from shapely.geometry import mapping, shape
from shapely.ops import transform as shp_transform
import pyproj
to3857 = pyproj.Transformer.from_crs(4326, 3857, always_xy=True).transform
foot = [shp_transform(to3857, shape(f["geometry"])) for f in plan["features"] if f["properties"]["tile_name"] in done]
govs = json.load(open(os.path.join(A.pub, "boundaries.geojson")))["features"]
gov_names = [g["properties"]["name"] for g in govs]
gov_geoms = [shp_transform(to3857, shape(g["geometry"])) for g in govs]
KRI = {"Erbil", "Duhok", "Sulaymaniyah", "Halabja"}
# our 2025-26 tiles, read at source so we can apply different "is it cereal?" rules
from rasterio.warp import transform_bounds
src = {}
for f in glob.glob(os.path.join(A.run, "compact", "*.tif")):
    t = os.path.basename(f)[:-4]
    if t in done: src[t] = ("compact", f)
for f in glob.glob(os.path.join(A.run, "batch_*", "*", "*.tif")):
    t = os.path.basename(os.path.dirname(f))
    if t in done: src[t] = ("raw", f)          # raw preferred when present (same result as compact)
tb = {}
for t, (kind, f) in src.items():
    with rasterio.open(f) as ds:
        tb[t] = transform_bounds(ds.crs, "EPSG:3857", *ds.bounds)
print(f"Processed tiles: {len(done)} ({sum(1 for k in src.values() if k[0] == 'raw')} raw, "
      f"{sum(1 for k in src.values() if k[0] == 'compact')} compact); published cells: {len(idx['cells'])}; comparing at zoom {Z}")
RULES = {"top_crop": "the most likely of 24 crops is wheat, barley, rye or oats (WorldCereal's default classification)",
         "p50": "P(wheat)+P(barley) ≥ 50% (a yes/no cereal decision, like the 2021 product)",
         "p70_all": "a cereal class with P(wheat)+P(barley) ≥ 70%, specks included",
         "p70": "a cereal class with P(wheat)+P(barley) ≥ 70%, in a patch of at least 0.1 ha "
                "(the map's default: the same 'confident cereal' the crops page shows)"}
MIN_PATCH_HA = 0.1


def small_patches(mask, transform, min_ha=MIN_PATCH_HA):
    """Pixels of `mask` that belong to 8-connected clusters smaller than min_ha (same filter as publish_crops.py)."""
    min_px = math.ceil(min_ha * 1e4 / abs(transform.a * transform.e))
    lab, n = ndimage.label(mask, structure=np.ones((3, 3), bool))
    if not n:
        return np.zeros(mask.shape, bool)
    size = np.bincount(lab.ravel())
    small = size < min_px
    small[0] = False
    return small[lab]


def read_tile(kind, f):
    """-> dict of 0/1 uint8 masks on the tile's own grid, plus crs/transform."""
    with rasterio.open(f) as ds:
        names = list(ds.descriptions)
        if kind == "compact":
            cls = ds.read(1); pw = ds.read(3).astype(np.int16); pb = ds.read(4).astype(np.int16)
            cl = json.loads(ds.tags()["classes"]); pro = np.zeros_like(pw)
        else:
            season = next(nm.split(":")[1] for nm in names if nm.startswith("croptype_classification:"))
            cl = [nm.split(":")[2] for nm in names if nm.startswith(f"croptype_probability:{season}:")]
            cls = ds.read(names.index(f"croptype_classification:{season}") + 1)
            P = lambda c: ds.read(names.index(f"croptype_probability:{season}:{c}") + 1).astype(np.int16) if c in cl else 0
            pw, pb = P("wheat"), P("barley"); pro = np.zeros_like(pw)   # wheat + barley only, as for compact tiles
        valid = cls != 255
        crop = cls < len(cl)
        winter = [i for i, c in enumerate(cl) if c in ("wheat", "barley", "rye", "oats")]
        nocrop = cl.index("no_crop") if "no_crop" in cl else -1
        psum = pw + pb + pro
        top = np.isin(cls, winter)
        p70_all = top & (psum >= 70)
        m = {"valid": valid,
             "top_crop": np.isin(cls, winter),
             "p50": crop & (psum >= 50), "p70_all": p70_all,
             "p70": p70_all & ~small_patches(p70_all, ds.transform),
             "wheat26": cls == cl.index("wheat"), "barley26": cls == cl.index("barley"),
             "crop26": crop & (cls != nocrop), "crop26_incl_fallow": crop}
        return {k: v.astype(np.uint8) for k, v in m.items()}, ds.crs, ds.transform

ts = 2 * R / 2 ** Z            # tile size (m)
res = ts / 256
# counters per governorate index (0 = outside all), in hectares
keys = ["wc21", "wc26", "wc_both", "crop21", "crop26", "crop_both", "crop26_incl_fallow", "wheat26", "barley26", "processed"] + \
       [f"{p}_{r}" for r in RULES for p in ("wc26", "wc_both")]
acc = defaultdict(lambda: defaultdict(float))
diff_index = []

for ci, c in enumerate(idx["cells"]):
    cz, cx, cy = c["z"], c["x"], c["y"]
    k = 2 ** (Z - cz)
    n = 256 * k
    tr = Affine(res, 0, -R + cx * k * ts, 0, -res, R - cy * k * ts)
    # our tiles → this grid; each mask by area fraction (≥ 50 % of the 16 m cell = yes)
    acc_f = {}
    for t, b in tb.items():
        if b[2] <= tr.c or b[0] >= tr.c + n * res or b[3] <= tr.f - n * res or b[1] >= tr.f:
            continue
        masks, crs, ttr = read_tile(*src[t])
        valid = None
        for name, m in masks.items():
            dst = np.full((n, n), np.nan, np.float32)
            reproject(m.astype(np.float32), dst, src_transform=ttr, src_crs=crs, dst_transform=tr, dst_crs="EPSG:3857",
                      resampling=Resampling.average, dst_nodata=np.nan)
            if name == "valid":
                valid = ~np.isnan(dst) & (dst > 0)
                continue
            a = acc_f.setdefault(name, np.full((n, n), np.nan, np.float32))
            put = np.isnan(a) & valid
            a[put] = dst[put]
        del masks
    if not acc_f:
        continue
    M = {k_: np.nan_to_num(v, nan=0.0) >= 0.5 for k_, v in acc_f.items()}
    del acc_f
    wc26 = M["top_crop"]; wheat26 = M["wheat26"]; barley26 = M["barley26"]
    crop26 = M["crop26"]; crop26_all = M["crop26_incl_fallow"]
    proc = rasterize([(mapping(g), 1) for g in foot], out_shape=(n, n), transform=tr, fill=0, dtype="uint8").astype(bool)
    if not proc.any():
        continue
    gid = rasterize([(mapping(g), i + 1) for i, g in enumerate(gov_geoms)], out_shape=(n, n), transform=tr, fill=0, dtype="int16")
    # 2021 masks from the WMTS
    m21 = {}
    jobs = {}
    with cf.ThreadPoolExecutor(A.threads) as ex:
        for p in PRODUCTS:
            m21[p] = np.zeros((n, n), bool)
            for dx in range(k):
                for dy in range(k):
                    # skip tiles with nothing processed
                    if not proc[dy * 256:(dy + 1) * 256, dx * 256:(dx + 1) * 256].any():
                        continue
                    jobs[ex.submit(fetch, p, Z, cx * k + dx, cy * k + dy)] = (p, dx, dy)
        for fu in cf.as_completed(jobs):
            p, dx, dy = jobs[fu]
            m = fu.result()
            if m is not None:
                m21[p][dy * 256:(dy + 1) * 256, dx * 256:(dx + 1) * 256] = m
    wc21, crop21 = m21["wintercereals"], m21["temporarycrops"]
    # pixel area (ha) per row
    ys = R - cy * k * ts - (np.arange(n) + 0.5) * res
    lat = np.arctan(np.sinh(ys / R * math.pi))
    ha_row = (res * np.cos(lat)) ** 2 / 1e4
    HA = np.broadcast_to(ha_row[:, None], (n, n))
    layers = {"wc21": wc21, "wc26": wc26, "wc_both": wc21 & wc26, "crop21": crop21, "crop26": crop26,
              "crop_both": crop21 & crop26, "crop26_incl_fallow": crop26_all, "wheat26": wheat26, "barley26": barley26,
              "processed": np.ones((n, n), bool)}
    for r in RULES:
        layers[f"wc26_{r}"] = M[r]; layers[f"wc_both_{r}"] = wc21 & M[r]
    g = gid[proc]
    hp = HA[proc]
    for key, L in layers.items():
        sums = np.bincount(g, weights=hp * L[proc], minlength=len(govs) + 1)
        for i, v in enumerate(sums):
            if v:
                acc[i][key] += float(v)
    # difference image (half resolution) for the preview map
    d = np.zeros((n, n), np.uint8)
    d[proc & wc21 & wc26] = 1
    d[proc & ~wc21 & wc26] = 2
    d[proc & wc21 & ~wc26] = 3
    d2 = d[::2, ::2]
    img = Image.frombuffer("P", (d2.shape[1], d2.shape[0]), np.ascontiguousarray(d2).tobytes(), "raw", "P", 0, 1)
    pal = [0, 0, 0, 46, 125, 50, 227, 181, 5, 123, 63, 160] + [0] * (256 * 3 - 12)
    img.putpalette(pal)
    fn = f"z{cz}_{cx}_{cy}.png"
    img.save(os.path.join(A.out, "diff", fn), optimize=True, transparency=bytes([0, 220, 220, 220] + [0] * 252))
    lon0, lat1 = (-R + cx * k * ts) / R * 180, math.degrees(math.atan(math.sinh((R - cy * k * ts) / R * math.pi)))
    lon1, lat0 = (-R + (cx + 1) * k * ts) / R * 180, math.degrees(math.atan(math.sinh((R - (cy + 1) * k * ts) / R * math.pi)))
    diff_index.append({"png": fn, "bounds": [[lat0, lon0], [lat1, lon1]]})
    print(f"  cell {ci + 1}/{len(idx['cells'])} z{cz}/{cx}/{cy}: 2021 winter cereals {wc21[proc].mean() * 100:4.1f}% · "
          f"2026 {wc26[proc].mean() * 100:4.1f}% (P≥50 {M['p50'][proc].mean() * 100:4.1f}%, P≥70 {M['p70'][proc].mean() * 100:4.1f}%) "
          f"of processed pixels · fetch failures so far {dict(failures)}")


def metrics(a):
    out = {k: round(a.get(k, 0.0)) for k in keys}
    wb, w1, w2 = a.get("wc_both", 0), a.get("wc21", 0), a.get("wc26", 0)
    cb, c1, c2 = a.get("crop_both", 0), a.get("crop21", 0), a.get("crop26", 0)
    out["wc_f1"] = round(2 * wb / (w1 + w2), 3) if w1 + w2 else None
    out["wc_2026_also_2021"] = round(wb / w2, 3) if w2 else None    # share of 2026 cereal that 2021 also mapped as cereal
    out["wc_2021_also_2026"] = round(wb / w1, 3) if w1 else None
    out["crop_f1"] = round(2 * cb / (c1 + c2), 3) if c1 + c2 else None
    out["wc_ratio_2026_to_2021"] = round(w2 / w1, 2) if w1 else None
    for r in RULES:
        w2r, wbr = a.get(f"wc26_{r}", 0), a.get(f"wc_both_{r}", 0)
        out[f"rule_{r}"] = {"wc26": round(w2r), "wc_both": round(wbr), "ratio": round(w2r / w1, 2) if w1 else None,
                            "f1": round(2 * wbr / (w1 + w2r), 3) if w1 + w2r else None}
    return out


rows = []
regions = defaultdict(lambda: defaultdict(float)); total = defaultdict(float)
for i, a in acc.items():
    name = gov_names[i - 1] if i else "(outside boundaries)"
    rows.append({"name": name, "region": ("Kurdistan Region" if name in KRI else "Other governorates") if i else "", **metrics(a)})
    reg = ("Kurdistan Region" if name in KRI else "Other governorates") if i else "Outside Iraq's boundary"
    for k_, v in a.items():
        regions[reg][k_] += v; total[k_] += v
rows.sort(key=lambda r: -r["processed"])
res_json = {
    "compared_at_zoom": Z, "pixel_m_ground_approx": round(res * math.cos(math.radians(33.5)), 1),
    "tiles_processed": len(done), "fetch_failures": dict(failures),
    "note": "Only inside processed tiles. 2021 = ESA WorldCereal 2021 v100 classification (wintercereals, temporarycrops). "
            "2026 = our run of the current WorldCereal model; winter cereals = wheat+barley+rye+oats; a 16 m cell counts when ≥50% of its 10 m pixels do. "
            "2021 was a severe drought year in Iraq, so less rain-fed land was cropped than in a normal year.",
    "rules": RULES, "default_rule": "p70",
    "total": metrics(total),
    "regions": {k_: metrics(v) for k_, v in regions.items()},
    "governorates": rows,
}
json.dump(res_json, open(os.path.join(A.out, "compare_2021.json"), "w"), indent=1)
json.dump({"tiles": diff_index, "legend": {"1": "cereal in both years", "2": "cereal in 2026 only", "3": "cereal in 2021 only"}},
          open(os.path.join(A.out, "diff", "index.json"), "w"))
with open(os.path.join(A.out, "compare_2021_governorates.csv"), "w", newline="", encoding="utf-8-sig") as fh:
    flat = [{k_: (json.dumps(v) if isinstance(v, dict) else v) for k_, v in r.items()} for r in rows]
    w = csv.DictWriter(fh, fieldnames=list(flat[0].keys())); w.writeheader(); w.writerows(flat)

t = res_json["total"]
print(f"\nInside the {len(done)} processed tiles ({t['processed']:,} ha):")
print(f"  Winter cereals 2021 (official): {t['wc21']:>10,} ha")
print(f"  Winter cereals 2026 (ours):     {t['wc26']:>10,} ha   (wheat {t['wheat26']:,} · barley {t['barley26']:,})")
print(f"  Mapped as cereal in both:       {t['wc_both']:>10,} ha   → {t['wc_2026_also_2021'] and round(t['wc_2026_also_2021'] * 100)}% of our 2026 cereal was also 2021 cereal")
print(f"  Cropland 2021 / 2026:           {t['crop21']:>10,} / {t['crop26']:,} ha (F1 {t['crop_f1']})")
print("\nBy region:")
for k_, m in res_json["regions"].items():
    print(f"  {k_}: cereal 2021 {m['wc21']:,} ha · 2026 {m['wc26']:,} ha · both {m['wc_both']:,} ha · ratio {m['wc_ratio_2026_to_2021']}")
print("\nBy governorate (processed area only):")
for r in rows:
    print(f"  {r['name']:16s} 2021 {r['wc21']:>9,} · 2026 {r['wc26']:>9,} · both {r['wc_both']:>9,} · F1 {r['wc_f1']}")
print("\nSensitivity — 2025-26 cereal under three rules (2021 is ESA's default product in every row):")
print(f"  {'rule':10s} {'area':22s} {'2025-26 ha':>12s} {'×2021':>7s} {'F1':>6s}")
for r in RULES:
    for nm, m in [("Iraq (processed)", res_json["total"])] + [(k_, v) for k_, v in res_json["regions"].items() if not k_.startswith("Outside")]:
        x = m[f"rule_{r}"]
        print(f"  {r:10s} {nm:22s} {x['wc26']:>12,} {x['ratio']!s:>7s} {x['f1']!s:>6s}")
if failures:
    print("\n⚠️  Some 2021 tiles failed to download:", dict(failures), "— those areas count as 'not cereal' in 2021. Re-run to retry.")


## Quick check: can we read the 2021 maps? (a few seconds)

In [ ]:
!python /content/compare_2021.py "$RUN" "$PUB" "$OUT" --test

## Full comparison

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
!python /content/compare_2021.py "$RUN" "$PUB" "$OUT"
stamp = f"{datetime.datetime.now(datetime.timezone.utc):%Y%m%d_%H%M}"
zip_path = shutil.make_archive(f"/content/compare_2021_{stamp}", "zip", OUT)
shutil.copy(zip_path, f"{RUN}/publish/")
shutil.copy(f"{OUT}/compare_2021.json", PUB)   # the crops page reads it from crops/2026/
print("compare_2021.json also copied into", PUB, "— upload it to crops/2026/ on GitHub")
print("Zip:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB) — also saved in Drive next to publish/")
from google.colab import files
files.download(zip_path)